# Part 1

## Task 1

In [31]:
# load table first.
import pandas as pd
listings_clean_raw = pd.read_csv("../listings_clean.csv")
listings_clean = listings_clean_raw.copy()

Mean and median, respectively: 7.6, 8.0

In [32]:
number_of_reviews = pd.Series([3, 5, 8, 10, 12])

print(number_of_reviews.mean(), number_of_reviews.median())

7.6 8.0


Adding the more extreme value will result in the median skewing very little, but the mean will increase a lot.
(221.8, 9.0).

In [38]:
number_of_reviews_updated = pd.Series([3, 5, 8, 10, 12, 1293])

print(number_of_reviews_updated.mean().round(1), number_of_reviews_updated.median())

221.8 9.0


Mean and median of the real column 'number_of_reviews'. (35.8, 9.0) 

In [40]:
print(listings_clean['number_of_reviews'].mean().round(1), listings_clean['number_of_reviews'].median())

35.8 9.0


In order to answer "How many reveiws does a typical listing have?", I'd look at the median, since it describes the 
most average values at the middle; that is most typical. That is, it is most typical to have 9.0 reviews or less.

## Task 2

Noteworthy values in the description: 
**count** - 4574.0
**mean** - 35.8
**std** - 78.4
**min** - 0.0
**25%** - 2.0
**50%** - 9.0
**75%** - 29.0
**max** - 1293.0

In [42]:
listings_clean['number_of_reviews'].describe().round(1)

count    4574.0
mean       35.8
std        78.4
min         0.0
25%         2.0
50%         9.0
75%        29.0
max      1293.0
Name: number_of_reviews, dtype: float64

Quantiles. 
**10th** - 0.0
**25th** - 2.0
**50th** - 9.0
**75th** - 29.0
**90th** - 94.0
**99th** - 400.0

In [45]:
listings_clean['number_of_reviews'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0)

0.10      0.0
0.25      2.0
0.50      9.0
0.75     29.0
0.90     94.0
0.99    400.0
Name: number_of_reviews, dtype: float64

Q1, Q3, and IQR. (2.0, 29.0, and 27.0)
IQR is the spread of the middle data (the 50%). 
It is more easily understood by drawing up Q1, Q2 and Q3 as a box, which shows that everything within this box is limited
by the sides (Q3 and Q1, 75% - 25% = 50%)

In [46]:
q1 = listings_clean['number_of_reviews'].quantile(0.25)
q3 = listings_clean['number_of_reviews'].quantile(0.75)

iqr = q3 - q1

print(f"Q1: {round(q1, 1)}, Q3: {round(q3, 1)}, IQR: {round(iqr, 1)}")

Q1: 2.0, Q3: 29.0, IQR: 27.0


Since 0.10/10% has 0.0 as its number, it means that at least the (first) 10% do not have any reviews.  

## Task 3

Upper and lower fences. (69.5 and -38.5, respectively)

In [47]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

print(f'upper: {upper.round(1)}, lower: {lower.round(1)}')

upper: 69.5, lower: -38.5


Logically, an outlier should not be able to go below 0. 
The minimum should be nothing, which is generally agreed upon to be 0, so 
a negative amount of 'number_of_reviews' would say how many reviews are lacking per listing.

582 entries over the upper fence (12.7%).

In [65]:
upper_entries = listings_clean[listings_clean['number_of_reviews'] > upper]

print(len(upper_entries))
print(round(((len(upper_entries) / len(listings_clean)) * 100), 1))

upper_entries['number_of_reviews'].describe().round(1)

582
12.7


count     582.0
mean      192.2
std       136.4
min        70.0
25%        98.0
50%       148.5
75%       236.5
max      1293.0
Name: number_of_reviews, dtype: float64

Count on room types. 
**Entire home/apt** - 446
**Private room** - 121
**Shared room** - 14
**Hotel room** - 1

In [67]:
upper_entries['room_type'].value_counts()

room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64

Median price. Should be 1459.5 and 1867.0, respectively.

In [73]:
print(f"Outliers: {upper_entries['price'].median().round(1)}, General: {listings_clean[listings_clean['number_of_reviews'] < upper]['price'].median()}")

Outliers: 1459.5, General: 1867.0


The consensus I come to is that the price is generally better on the ones with a higher amount of reviews.

## Task 4

Outliers. (582)

In [ ]:
listings_clean['reviews_outlier'] = listings_clean['number_of_reviews'] > upper

print(listings_clean['reviews_outlier'].sum())

582


New table with 
**count** (4574 and 3992), 
**mean** (35.8 and 12.9),
**median** (9.0 and 7.0) and 
**std** (78.4 and 15.5)
for **number_of_reviews**.

In [86]:
review_comparison = pd.DataFrame({
    'all': listings_clean['number_of_reviews'].agg(['count', 'mean', 'median', 'std']).round(1),
    'without_outliers': listings_clean.loc[~listings_clean['reviews_outlier'], 'number_of_reviews'].agg(['count', 'mean', 'median', 'std']).round(1)
})

review_comparison

,all,without_outliers
count,4574.0,3992.0
mean,35.8,12.9
median,9.0,7.0
std,78.4,15.5


Values should be: 
[**count** : -13.0, 
**mean** : -64.0,
**median** : -22.0,
**std** : -80.0]

The standard deviation changes the most, and the count the least if considered, otherwise the median. 

In [95]:
percentages = (((review_comparison['without_outliers'] - review_comparison['all']) / (review_comparison['all'])) * 100).round(0)

percentages

count    -13.0
mean     -64.0
median   -22.0
std      -80.0
dtype: float64

## Task 5

Description for review numbers per room type.
Shared rooms have the highest median, at 82.0.

[**Entire home/apt**: count - 3760, median - 8]
[**Hotel room**: count - 3, median - 56]
[**Private room**: count - 787, median - 12]
[**Shared room**: count - 24, median - 82]

In [98]:
listings_clean.groupby('room_type')['number_of_reviews'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
room_type,,,,,,,,
Entire home/apt,3760.0,34.0,76.0,0.0,2.0,8.0,27.0,1293.0
Hotel room,3.0,61.0,16.0,49.0,52.0,56.0,68.0,79.0
Private room,787.0,41.0,84.0,0.0,2.0,12.0,38.0,867.0
Shared room,24.0,114.0,122.0,0.0,28.0,82.0,146.0,510.0


If one has to not have anything said about, it would be Hotel room due to the low count. They are in comparison to the others, even if Shared room has very few, 
a little less significant YET.

## Task 6

Correlation between number_of_reviews and an expected selection of columns.

Should give the following correlations:
[**price** : -0.09,
**accommodates** : -0.05,
**bedrooms** : -0.14,
**availability_365** : 0.18,
**review_scores_rating** : -0.04,
**minimum_nights** : -0.07,
**day_since_review** : -0.19
]

In [ ]:
columns = ['price', 'accommodates', 'bedrooms', 'availability_365', 'review_scores_rating', 'minimum_nights', 'days_since_review', 'number_of_reviews']

listings_clean[columns].corr()['number_of_reviews'].round(2)

price                  -0.09
accommodates           -0.05
bedrooms               -0.14
availability_365        0.18
review_scores_rating   -0.04
minimum_nights         -0.07
days_since_review      -0.19
number_of_reviews       1.00
Name: number_of_reviews, dtype: float64

Strongest connection is -0.19 on days since review, so since it is a negative trend, it means: "The more reviews, the fewer the days are since a review."

Variable with no outliers, and the difference.
0.183 vs. 0.002

In [110]:
no_review_outlier = listings_clean[~listings_clean['reviews_outlier']] 

print(listings_clean['number_of_reviews'].corr(listings_clean['availability_365']).round(3))
print(listings_clean['number_of_reviews'].corr(no_review_outlier['availability_365']).round(3))

0.183
0.002


First of all, the first alternative does not have a high correlation.
However, since the outliers had very high values, it means that when we delete these, we see that these basically pulled the weight for the connection to 
availability_365 having more "matching" changes/values.
It becomes harder to see or seemingly like there being no correlation at all when the listings with a high number of reviews are omitted. 